In [10]:

# CELL 1:GPU CHECK

import os
import time
import cv2
import torch
from ultralytics import YOLO


print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Training will be slow.")

print(f"\nCurrent working folder (everything will save here): {os.getcwd()}")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
Active GPU: NVIDIA GeForce RTX 5080 Laptop GPU

Current working folder (everything will save here): c:\Users\akaka\Desktop\final 2\final 3\final 4


In [11]:

# CELL 2: FIND ALL DATASETS AUTOMATICALLY 


BASE_DIR = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\datasets"
SPLIT_NAMES = {"train", "valid", "test"}
SKIP_DIRS = {"combined_lists"}  # our own output folder, not a dataset

DATASETS = {}

for root, dirs, files in os.walk(BASE_DIR):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

    if os.path.basename(root) != "images":
        continue

    split_dir = os.path.dirname(root)           # .../<dataset>/<split>
    split_name = os.path.basename(split_dir)     # train / valid / test
    dataset_dir = os.path.dirname(split_dir)      # .../<dataset>

    if split_name not in SPLIT_NAMES:
        continue

    labels_dir = os.path.join(split_dir, "labels")
    dataset_name = os.path.relpath(dataset_dir, BASE_DIR)

    DATASETS.setdefault(dataset_name, {})[split_name] = (root, labels_dir)

print(f"Auto-discovered {len(DATASETS)} datasets:")
for name, splits in DATASETS.items():
    print(f"  {name}: {sorted(splits.keys())}")

Auto-discovered 5 datasets:
  gravel: ['test', 'train', 'valid']
  HRP4K\HRP4K: ['test', 'train', 'valid']
  New pothole detection.v2i.yolo26: ['test', 'train', 'valid']
  night_time: ['test', 'train', 'valid']
  Pothole.v1i.yolo26: ['test', 'train', 'valid']


In [12]:

# CELL 3: FIND WHICH IMAGES ARE ANNOTATED (POSITIVE VS NEGATIVE)


VALID_EXTS = (".jpg", ".jpeg", ".png")

def scan_split(images_dir, labels_dir):
    positive, negative = [], []
    if not os.path.exists(images_dir):
        return positive, negative

    for fname in os.listdir(images_dir):
        if not fname.lower().endswith(VALID_EXTS):
            continue
        img_path = os.path.join(images_dir, fname)
        label_path = os.path.join(labels_dir, os.path.splitext(fname)[0] + ".txt")

        has_content = False
        if os.path.exists(label_path):
            with open(label_path, "r", encoding="utf-8") as f:
                has_content = any(line.strip() for line in f)

        if has_content:
            positive.append(img_path)
        else:
            negative.append(img_path)

    return positive, negative


scan_results = {}

print(f"{'Dataset':<35}{'Split':<8}{'Positive':<10}{'Negative':<10}{'Total':<8}")

for ds_name, splits in DATASETS.items():
    scan_results[ds_name] = {}
    for split_name, (img_dir, lbl_dir) in splits.items():
        pos, neg = scan_split(img_dir, lbl_dir)
        scan_results[ds_name][split_name] = (pos, neg)
        print(f"{ds_name:<35}{split_name:<8}{len(pos):<10}{len(neg):<10}{len(pos)+len(neg):<8}")

Dataset                            Split   Positive  Negative  Total   
gravel                             test    373       289       662     
gravel                             train   3067      2000      5067    
gravel                             valid   756       589       1345    
HRP4K\HRP4K                        test    600       300       900     
HRP4K\HRP4K                        train   2803      1400      4203    
HRP4K\HRP4K                        valid   600       300       900     
New pothole detection.v2i.yolo26   test    1054      1         1055    
New pothole detection.v2i.yolo26   train   6087      4         6091    
New pothole detection.v2i.yolo26   valid   2094      0         2094    
night_time                         test    372       0         372     
night_time                         train   2571      0         2571    
night_time                         valid   742       0         742     
Pothole.v1i.yolo26                 test    275       2         2

In [13]:

# CELL 4: BUILD COMBINED TRAIN/VAL/TEST LISTS 


OUTPUT_DIR = os.path.join(os.getcwd(), "combined_lists")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def write_list(split_name, out_filename):
    all_paths = []
    for ds_name in DATASETS:
        pos, neg = scan_results[ds_name][split_name]
        all_paths.extend(pos)
        all_paths.extend(neg)

    out_path = os.path.join(OUTPUT_DIR, out_filename)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_paths))
    print(f"{split_name}: wrote {len(all_paths)} image paths -> {out_path}")
    return out_path

train_txt = write_list("train", "train.txt")
valid_txt = write_list("valid", "valid.txt")
test_txt  = write_list("test", "test.txt")

train: wrote 19871 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\train.txt
valid: wrote 5636 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\valid.txt
test: wrote 3266 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\test.txt


In [14]:

# CELL 5: WRITE THE FINAL data.yaml

import yaml

data_yaml_path = os.path.join(OUTPUT_DIR, "final4_data.yaml")

data_yaml = {
    "train": train_txt,
    "val": valid_txt,
    "test": test_txt,
    "nc": 1,
    "names": ["pothole"],
}

with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {data_yaml_path}")

data.yaml written to: c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\final4_data.yaml


In [15]:

# CELL 6: CLASS VERIFICATION


def verify_classes():
    invalid = 0
    checked = 0
    for ds_name in DATASETS:
        for split_name in ["train", "valid", "test"]:
            pos, _ = scan_results[ds_name][split_name]
            lbl_dir = DATASETS[ds_name][split_name][1]
            for img_path in pos:
                lbl_path = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
                with open(lbl_path, "r", encoding="utf-8") as f:
                    for line in f:
                        if not line.strip():
                            continue
                        checked += 1
                        class_id = int(line.split()[0])
                        if class_id != 0:
                            invalid += 1

    print(f"Checked {checked} annotations across all datasets.")
    if invalid == 0:
        print("Verification passed: every annotation is class 0 (pothole).")
    else:
        print(f"WARNING: found {invalid} annotations with a class other than 0!")

verify_classes()

Checked 57084 annotations across all datasets.
Verification passed: every annotation is class 0 (pothole).


In [ ]:

# CELL 7: PHASE 1  FROZEN BACKBONE (15 EPOCHS)


PROJECT_NAME = "Pothole_Final4_Runs"  # everything saves under ./runs/detect/<this name>/...

model_phase1 = YOLO("yolo26m.pt")

print("--- PHASE 1: training with the backbone frozen (15 epochs) ---")
phase1_results = model_phase1.train(
    data=data_yaml_path,
    epochs=15,
    patience=100,     # don't early-stop during this short warm-up phase
    imgsz=640,

    # Hardware settings tuned for RTX 5080 Laptop / 32GB RAM
    device=0,
    cache="disk",
    workers=6,
    batch=24,

    # Fine-tuning (frozen backbone) - see comment above
    freeze=10,
    optimizer="AdamW",
    lr0=0.001,
    warmup_epochs=1.0,

    # Augmentation (helps with day/night/gravel lighting variety)
    hsv_v=0.4,
    hsv_s=0.3,
    mosaic=1.0,

    project=PROJECT_NAME,
    name="phase1_frozen_backbone",
)

# This is where Phase 1's best weights ended up - Phase 2 continues from here.
phase1_best_path = os.path.join("runs", "detect", PROJECT_NAME, "phase1_frozen_backbone", "weights", "best.pt")
print(f"\nPhase 1 finished. Best weights saved at: {phase1_best_path}")

--- PHASE 1: training with the backbone frozen (15 epochs) ---
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\final4_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.3, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, m

In [ ]:

# CELL 8: PHASE 2 - FINE-TUNE THE WHOLE NETWORK (85 EPOCHS)


model_phase2 = YOLO(phase1_best_path)

print("--- PHASE 2: fine-tuning the full network (85 epochs) ---")
phase2_results = model_phase2.train(
    data=data_yaml_path,
    epochs=85,
    patience=15,      # normal early stopping now that we're in the real training phase
    imgsz=640,

    device=0,
    cache="disk",
    workers=6,
    batch=24,

    # freeze is left out entirely here - nothing is frozen, the whole
    # network (backbone + neck + head) can now update.
    optimizer="AdamW",
    lr0=0.0001,       # lower than Phase 1 - careful fine-tuning, not big changes
    warmup_epochs=1.0,

    hsv_v=0.4,
    hsv_s=0.3,
    mosaic=1.0,

    project=PROJECT_NAME,
    name="phase2_finetune_all",
)

best_model_path = os.path.join("runs", "detect", PROJECT_NAME, "phase2_finetune_all", "weights", "best.pt")
print(f"\nPhase 2 finished. FINAL best weights saved at: {best_model_path}")

--- PHASE 2: fine-tuning the full network (85 epochs) ---
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\akaka\Desktop\final 2\final 3\final 4\combined_lists\final4_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=85, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.3, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max

In [16]:

# CELL 9: ACCURACY SUMMARY 


best_model_path = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.pt"
final_model = YOLO(best_model_path)
metrics = final_model.val(data=data_yaml_path, split="val")

print(" FINAL MODEL ACCURACY ")
print(f"Precision : {metrics.box.mp:.3f}  ")#(of everything the model flagged as a pothole, how much really was one)
print(f"Recall    : {metrics.box.mr:.3f}  ")#(of all real potholes, how many the model actually found)
print(f"mAP50     : {metrics.box.map50:.3f} ")# (overall accuracy at a lenient overlap threshold the main headline number)
print(f"mAP50-95  : {metrics.box.map:.3f}  ")#(overall accuracy averaged across strict-to-lenient overlap a tougher, more complete measure)

Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 125.915.1 MB/s, size: 80.7 KB)
val: Scanning C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\HRP4K\HRP4K\valid\labels.cache... 5636 images, 893 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 5636/5636  0.0s
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\valid\images\20250507080326_0060_frame_00010_jpg.rf.75e1e74655db728d494688456edddaa8.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\valid\images\signal-2025-06-25-183557_frame_00075_jpg.rf.449477ce63cf9106d88f1857ad790155.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\valid\images\signal-202

In [9]:

# CELL 10: SPEED BENCHMARK (FPS)


import time
import glob

sample_images = glob.glob(os.path.join(OUTPUT_DIR, "*.txt"))  # placeholder, replaced below
with open(test_txt, "r", encoding="utf-8") as f:
    all_test_images = [line.strip() for line in f if line.strip()]

benchmark_images = all_test_images[:100]  # time 100 images, plenty for a stable average
print(f"Benchmarking speed on {len(benchmark_images)} images...")

# one "warm-up" prediction first - the very first call is always slower
# because the GPU is still loading things into memory
_ = final_model.predict(source=benchmark_images[0], imgsz=640, verbose=False)

start_time = time.time()
for img_path in benchmark_images:
    _ = final_model.predict(source=img_path, imgsz=640, verbose=False)
elapsed = time.time() - start_time

fps = len(benchmark_images) / elapsed
print(f"\nProcessed {len(benchmark_images)} images in {elapsed:.2f} seconds")
print(f"Speed: {fps:.1f} FPS  (frames per second)")
print(f"For reference: 15-30 FPS is generally considered smooth for live video.")

Benchmarking speed on 100 images...

Processed 100 images in 0.99 seconds
Speed: 100.7 FPS  (frames per second)
For reference: 15-30 FPS is generally considered smooth for live video.


In [10]:

# CELL 11: DAY vs NIGHT TEST EVALUATION


def write_subset_list(dataset_names, split_name, out_filename):
    paths = []
    for ds_name in dataset_names:
        pos, neg = scan_results[ds_name][split_name]
        paths.extend(pos)
        paths.extend(neg)
    out_path = os.path.join(OUTPUT_DIR, out_filename)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(paths))
    return out_path, len(paths)

all_dataset_names = list(DATASETS.keys())
night_datasets = [name for name in all_dataset_names if "night" in name.lower()]
day_datasets = [name for name in all_dataset_names if name not in night_datasets]

print(f"Treating as DAY: {day_datasets}")
print(f"Treating as NIGHT: {night_datasets}")

day_test_txt, day_count = write_subset_list(day_datasets, "test", "day_test.txt")
night_test_txt, night_count = write_subset_list(night_datasets, "test", "night_test.txt")
print(f"\nDay test images: {day_count}, Night test images: {night_count}")

def make_eval_yaml(test_txt_path, out_name):
    yaml_path = os.path.join(OUTPUT_DIR, out_name)
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(
            {"train": train_txt, "val": valid_txt, "test": test_txt_path, "nc": 1, "names": ["pothole"]},
            f, sort_keys=False,
        )
    return yaml_path

day_yaml = make_eval_yaml(day_test_txt, "day_eval.yaml")
night_yaml = make_eval_yaml(night_test_txt, "night_eval.yaml")

print("DAYTIME PERFORMANCE ")
day_metrics = final_model.val(data=day_yaml, split="test")
print(f"Day mAP50: {day_metrics.box.map50:.4f}")

print(" NIGHTTIME PERFORMANCE ")
night_metrics = final_model.val(data=night_yaml, split="test")
print(f"Night mAP50: {night_metrics.box.map50:.4f}")

Treating as DAY: ['gravel', 'HRP4K\\HRP4K', 'New pothole detection.v2i.yolo26', 'Pothole.v1i.yolo26']
Treating as NIGHT: ['night_time']

Day test images: 2894, Night test images: 372
DAYTIME PERFORMANCE 
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
val: Fast image access  (ping: 0.50.1 ms, read: 122.2144.3 MB/s, size: 479.9 KB)
val: Scanning C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\HRP4K\HRP4K\test\labels... 2894 images, 592 backgrounds, 4 corrupt: 100% ━━━━━━━━━━━━ 2894/2894 1.3Kit/s 2.3s0.1s
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\New pothole detection.v2i.yolo26\test\images\img-179_jpg.rf.5861bc35f91da5a60d6ffefc00851617.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\test\images\signal-2025-06-26-111944_frame_00085_jpg.rf.2a1cd758664bab401ceda0d6e1bf4098.jpg: ignoring corrupt image/label: lab

In [12]:

# CELL 12: LIVE WEBCAM  (POTHOLE)


CAMERA_INDEX = 1  # 0 = laptop webcam, 1 = phone via Phone Link

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Could not open camera. Check CAMERA_INDEX, or that Phone Link is connected.")
else:
    print("Camera started (original model). Click the video window and press 'q' to stop.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = final_model.predict(frame, imgsz=640, conf=0.35, verbose=False)

        for result in results:
            if result.boxes is None:
                continue
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(frame, f"Pothole {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        cv2.imshow("Pothole Detection - press q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Camera stopped.")

Camera started (original model). Click the video window and press 'q' to stop.
Camera stopped.


In [ ]:

# CELL 13: COMPRESS THE MODEL (FP16 ONNX)


import os

print("Exporting Final 4 to FP16 ONNX")
compressed_model_path = final_model.export(
    format="onnx",
    half=True,    # This triggers FP16 Quantization / Compression
    imgsz=640,
    simplify=True # Optimizes the model architecture for speed
)

original_size = os.path.getsize(best_model_path) / (1024 * 1024)
compressed_size = os.path.getsize(compressed_model_path) / (1024 * 1024)


print(f"Original size:   {original_size:.1f} MB")
print(f"Compressed size: {compressed_size:.1f} MB")
print(f"Total Reduction: {(1 - compressed_size/original_size) * 100:.1f}%")
print(f"Saved to: {compressed_model_path}")

Exporting to ONNX FP16. This performs both Quantization (32-bit to 16-bit) and Compression...
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CPU (Intel Core Ultra 9 275HX)

PyTorch: starting from 'C:\Users\akaka\Desktop\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.0 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: converting to FP16...
ONNX: export success  3.6s, saved as 'C:\Users\akaka\Desktop\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.onnx' (39.0 MB)

Export complete (4.0s)
Results saved to C:\Users\akaka\Desktop\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\akaka\Desk

In [16]:

# CELL 14: VALIDATE THE COMPRESSED MODEL


compressed_model = YOLO(compressed_model_path)
comp_metrics = compressed_model.val(data=data_yaml_path, split="val", imgsz=640, device="cpu")

print("ORIGINAL vs COMPRESSED ")
print(f"{'Metric':<12}{'Original':<12}{'Compressed':<12}{'Difference':<12}")
print(f"{'Precision':<12}{metrics.box.mp:<12.3f}{comp_metrics.box.mp:<12.3f}{comp_metrics.box.mp - metrics.box.mp:+.3f}")
print(f"{'Recall':<12}{metrics.box.mr:<12.3f}{comp_metrics.box.mr:<12.3f}{comp_metrics.box.mr - metrics.box.mr:+.3f}")
print(f"{'mAP50':<12}{metrics.box.map50:<12.3f}{comp_metrics.box.map50:<12.3f}{comp_metrics.box.map50 - metrics.box.map50:+.3f}")
print(f"{'mAP50-95':<12}{metrics.box.map:<12.3f}{comp_metrics.box.map:<12.3f}{comp_metrics.box.map - metrics.box.map:+.3f}")

Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CPU (Intel Core Ultra 9 275HX)
Loading C:\Users\akaka\Desktop\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING requirements:  Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Deskt

In [18]:

# CELL 15: SPEED BENCHMARK - COMPRESSED MODEL


import time

with open(test_txt, "r", encoding="utf-8") as f:
    all_test_images = [line.strip() for line in f if line.strip()]

benchmark_images = all_test_images[:100]
print(f"Benchmarking COMPRESSED model speed on {len(benchmark_images)} images...")

# Warm-up run 
_ = compressed_model.predict(source=benchmark_images[0], imgsz=640, verbose=False, device="cpu")

start_time = time.time()
for img_path in benchmark_images:
    _ = compressed_model.predict(source=img_path, imgsz=640, verbose=False, device="cpu")
elapsed = time.time() - start_time

compressed_fps = len(benchmark_images) / elapsed
print(f"\nProcessed {len(benchmark_images)} images in {elapsed:.2f} seconds")
print(f"Compressed model speed: {compressed_fps:.1f} FPS")

Benchmarking COMPRESSED model speed on 100 images...
Loading C:\Users\akaka\Desktop\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING requirements:  Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scri

In [18]:
# FINAL 4 - SOURCE-BY-SOURCE TEST EVALUATION
# No training. Reads existing test data only.

import os
import yaml
import tempfile
from ultralytics import YOLO

# Final 4 trained model
best_model_path = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\runs\detect\Pothole_Final4_Runs\phase2_finetune_all\weights\best.pt"

# Check that Cell 2 has been run
required_vars = ["DATASETS", "scan_results"]
missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Run Cell 2 first. Missing: {missing}"
    )

model = YOLO(best_model_path)

results_table = []

with tempfile.TemporaryDirectory(prefix="Final4_Source_Test_") as temp_dir:

    for i, ds_name in enumerate(DATASETS.keys(), start=1):

        positive, negative = scan_results[ds_name]["test"]
        test_images = positive + negative

        if not test_images:
            continue

        test_txt = os.path.join(temp_dir, f"source_{i}_test.txt")

        with open(test_txt, "w", encoding="utf-8") as f:
            f.write("\n".join(test_images))

        eval_yaml = os.path.join(temp_dir, f"source_{i}.yaml")

        with open(eval_yaml, "w", encoding="utf-8") as f:
            yaml.safe_dump(
                {
                    "train": test_txt,
                    "val": test_txt,
                    "test": test_txt,
                    "nc": 1,
                    "names": ["pothole"]
                },
                f,
                sort_keys=False
            )

        print("\n" + "=" * 65)
        print(f"Testing: {ds_name}")
        print(f"Test images: {len(test_images)}")
        print("=" * 65)

        metrics = model.val(
            data=eval_yaml,
            split="test",
            imgsz=640,
            batch=24,
            device=0,
            workers=0,
            plots=False,
            save=False,
            verbose=False,
            project=temp_dir,
            name=f"eval_{i}"
        )

        results_table.append([
            ds_name,
            len(test_images),
            float(metrics.box.mp),
            float(metrics.box.mr),
            float(metrics.box.map50),
            float(metrics.box.map)
        ])


print("\nFINAL 4 - PERFORMANCE BY DATASET SOURCE\n")

print(
    f"{'Dataset':<38}"
    f"{'Images':>8}"
    f"{'Precision':>12}"
    f"{'Recall':>10}"
    f"{'mAP50':>10}"
    f"{'mAP50-95':>12}"
)

print("-" * 90)

for name, images, p, r, m50, m5095 in results_table:
    print(
        f"{name:<38}"
        f"{images:>8}"
        f"{p:>12.3f}"
        f"{r:>10.3f}"
        f"{m50:>10.3f}"
        f"{m5095:>12.3f}"
    )


Testing: gravel
Test images: 662
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 821.8287.5 MB/s, size: 107.2 KB)
val: Scanning C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\test\labels... 662 images, 289 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 662/662 4.4Kit/s 0.1s0.2s
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\test\images\signal-2025-06-26-111944_frame_00085_jpg.rf.2a1cd758664bab401ceda0d6e1bf4098.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\datasets\gravel\test\images\signal-2025-06-26-111944_frame_00085_jpg.rf.ba28be5928d27fbab9b72f61131534df.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\akaka\Desktop\final 2\final 3\final 4\dat